# 3 · Qualitative scores on the free-text fields

This notebook produces the results of **Section "Qualitative scores on free-text fields"**
and of the appendix **"Cohen's κ for the Likert dimensions"**:

| Output in this notebook | Paper table |
|---|---|
| Mean scores per annotator, exact / within-1 agreement, Gwet's AC2, LLM average | Table `tab:quality_scores` |
| Weighted Cohen's κ and Pearson r per dimension (incl. citation reasons) | Appendix Table `tab:quality_scores_kappa` |

**Setup.** Nine dimensions are rated on 1–5 Likert scales (two on the issue `<text>`,
four on `<summary>`, one each on `<factual_premises>`, `<judge_reasoning>`,
`<issue_outcome>`). Inter-annotator agreement is computed on the 20 shared judgments
(issues marked present by both annotators); the LLM average is computed on the full
corpus of 50 judgments, with shared issues contributing the mean of the two annotators'
scores.

**Why AC2 and not κ as headline statistic.** Cohen's κ is unstable under high prevalence /
marginal skew (the "kappa paradox", Feinstein & Cicchetti 1990; Wongpakaran et al. 2013).
On most dimensions both annotators give 4 or 5 to nearly every issue, which collapses the
chance-correction term and can produce near-zero or negative κ that does not reflect
substantive disagreement. Gwet's AC2 (Gwet 2014) is designed to be robust to this regime;
κ values are reported for completeness in the appendix table.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DATA = '../data'

# Expert annotations of the LLM extraction on the 50 test judgments
# (one row per LLM-extracted issue). Annotator mapping used in the paper:
#   A1 = Alessia, A2 = Piera.
df_a = pd.read_csv(f'{DATA}/validation_annotator_A1.csv')   # A1
df_p = pd.read_csv(f'{DATA}/validation_annotator_A2.csv')   # A2

sentenze_a = set(df_a['Sentenza'])
sentenze_p = set(df_p['Sentenza'])
common_sentenze = sentenze_a & sentenze_p

print(f"A1: {len(df_a)} extracted issues over {df_a['Sentenza'].nunique()} judgments")
print(f"A2: {len(df_p)} extracted issues over {df_p['Sentenza'].nunique()} judgments")
print(f"Judgments annotated by both (shared set): {len(common_sentenze)}")

A1: 54 extracted issues over 35 judgments
A2: 54 extracted issues over 35 judgments
Judgments annotated by both (shared set): 20


In [2]:
# ── Column-name constants (the CSV headers are in Italian) ──────────────────
COL_PRESENT = 'questione presente nella sentenza (TRUE/FALSE)'
COL_NOT_EXTRACTED = ("# questioni presenti NON estratte (numero minimo di questioni presenti "
                     "che in aggiunta a quelle estratte coprirebbero l'intero contenuto della sentenza)")

COL_GENERALITA = 'Score livello generalità della questione (1-5)'
COL_LINGUAGGIO = 'Score linguaggio giuridico (1-5)'
COL_CORRECTNESS = 'Correctness 1-5 (absence of elements that are not present in the source document)'
COL_FORM = ('Form 1-5 (coherence, readability, syntactic and grammatical correctness, '
            'and adherence to legal terminology)')
COL_COMPLETENESS = next(c for c in df_a.columns if c.startswith('Completeness'))
COL_SAT_RIASSUNTO    = 'Satisfaction 1-5 (degree of satisfaction with the global quality of the summary)'
COL_SAT_FATTO        = 'Satisfaction 1-5 (degree of satisfaction with the global quality of the facts)'
COL_SAT_RAGIONAMENTO = 'Satisfaction 1-5 (degree of satisfaction with the global quality of the reasoning)'
COL_SAT_ESITO        = 'Satisfaction 1-5 (degree of satisfaction with the global quality of the outcome formulation)'

COL_MOTIVI_ESTRATTI = '# motivi citazione estratti'
COL_SCORE_MOTIVI    = ('somma score motivi citazione (per ogni motivo 1 se il motivo corrisponde '
                       'a quanto detto nella sentenza (ovvero se il motivo è contenuto nella '
                       'sentenza e se corrisponde con quanto scritto) e 0 altrimenti)')

# (display name in the paper, column) — same order as Table tab:quality_scores
DIMENSIONS = [
    ('Issue <text> generality',          COL_GENERALITA),
    ('Issue <text> legal language',      COL_LINGUAGGIO),
    ('<summary> correctness',            COL_CORRECTNESS),
    ('<summary> form',                   COL_FORM),
    ('<summary> completeness',           COL_COMPLETENESS),
    ('<summary> satisfaction',           COL_SAT_RIASSUNTO),
    ('<factual_premises> satisfaction',  COL_SAT_FATTO),
    ('<judge_reasoning> satisfaction',   COL_SAT_RAGIONAMENTO),
    ('<issue_outcome> satisfaction',     COL_SAT_ESITO),
]
score_columns = [c for _, c in DIMENSIONS]

In [3]:
# ── Joint-present rule ────────────────────────────────────────────────────────
# An LLM-extracted issue enters the downstream metrics only if every annotator
# who saw it marked it as present in the judgment. For the 20 shared judgments
# this requires BOTH annotators to have marked it present; for the 30
# single-annotator judgments it reduces to that annotator's own label.
shared_concat_for_joint = pd.concat([
    df_a[df_a['Sentenza'].isin(common_sentenze)],
    df_p[df_p['Sentenza'].isin(common_sentenze)],
], ignore_index=True)
shared_concat_for_joint[COL_PRESENT] = shared_concat_for_joint[COL_PRESENT].astype(bool)

joint_present_lookup = (
    shared_concat_for_joint
    .groupby(['Sentenza', 'Questioni estratte'])[COL_PRESENT]
    .all()
)


def with_joint_present(df, common, lookup):
    """Add an `is_present_joint` column to `df`.

    For rows in shared judgments: AND of the two annotators' presence labels
    (an issue is kept only if *both* annotators marked it present).
    For rows in single-annotator judgments: that annotator's own label.
    """
    out = df.copy()
    out['is_present_joint'] = out[COL_PRESENT].astype(bool)
    is_shared = out['Sentenza'].isin(common)
    if is_shared.any():
        keys = pd.MultiIndex.from_arrays([
            out.loc[is_shared, 'Sentenza'].values,
            out.loc[is_shared, 'Questioni estratte'].values,
        ])
        out.loc[is_shared, 'is_present_joint'] = lookup.reindex(keys).values
    return out


df_a = with_joint_present(df_a, common_sentenze, joint_present_lookup)
df_p = with_joint_present(df_p, common_sentenze, joint_present_lookup)
print("is_present_joint flag added.")

is_present_joint flag added.


In [4]:
# ── Combined per-issue frame for the full corpus (N=50) ───────────────────────
def build_combined_df(df_a, df_p, common):
    """Build the per-issue frame used for the Overall (N=50) scores.

    - 30 unique-annotator judgments (15 + 15): rows kept as-is.
    - 20 shared judgments: numeric columns averaged per
      (Sentenza, Questione) across the two annotators.
      `is_present_joint` is identical across the two rows of a shared issue
      (it is the joint AND), so its mean equals that flag.
    """
    df_unique = pd.concat([
        df_a[~df_a['Sentenza'].isin(common)],
        df_p[~df_p['Sentenza'].isin(common)],
    ], ignore_index=True)

    df_shared_concat = pd.concat([
        df_a[df_a['Sentenza'].isin(common)],
        df_p[df_p['Sentenza'].isin(common)],
    ], ignore_index=True)
    df_shared_concat[COL_PRESENT]        = df_shared_concat[COL_PRESENT].astype(int)
    df_shared_concat['is_present_joint'] = df_shared_concat['is_present_joint'].astype(int)

    numeric_cols = df_shared_concat.select_dtypes(include='number').columns.tolist()

    df_shared_avg = (
        df_shared_concat
        .groupby(['Sentenza', 'Questioni estratte'], as_index=False, sort=False)
        .agg({c: 'mean' for c in numeric_cols})
    )

    # COL_NOT_EXTRACTED is a judgment-level field. The per-(Sent, Quest)
    # mean above is unsafe because A1 and A2 sometimes record it on
    # different Questione rows, so a later .max() per judgment would pick
    # the larger of the two instead of their average. Recompute at the
    # judgment level and place it on a single row per judgment.
    a_ne = df_a.groupby('Sentenza')[COL_NOT_EXTRACTED].max().fillna(0)
    p_ne = df_p.groupby('Sentenza')[COL_NOT_EXTRACTED].max().fillna(0)
    shared_ne_avg = (a_ne.loc[list(common)] + p_ne.loc[list(common)]) / 2

    df_shared_avg[COL_NOT_EXTRACTED] = pd.NA
    first_idx = df_shared_avg.drop_duplicates(subset='Sentenza', keep='first').index
    df_shared_avg.loc[first_idx, COL_NOT_EXTRACTED] = (
        df_shared_avg.loc[first_idx, 'Sentenza'].map(shared_ne_avg).values
    )

    return pd.concat([df_unique, df_shared_avg], ignore_index=True)


df_combined = build_combined_df(df_a, df_p, common_sentenze)
assert df_combined['Sentenza'].nunique() == 50, \
    f"Expected 50 judgments in df_combined, got {df_combined['Sentenza'].nunique()}"
print(f"Combined frame: {len(df_combined)} issues over "
      f"{df_combined['Sentenza'].nunique()} judgments")

Combined frame: 78 issues over 50 judgments


## Agreement statistics

`gwet_ac2` implements Gwet's AC2 for two raters on ordinal data (Gwet 2014, *Handbook of
Inter-Rater Reliability*, 4th ed., ch. 3), with the full declared rating scale used for
the chance correction even when the observed ratings are saturated in a sub-range.

In [5]:
import warnings
from sklearn.metrics import cohen_kappa_score

# Outcome satisfaction is rated 5 by both annotators on every issue, which makes
# kappa undefined (NaN); silence the corresponding sklearn warnings.
warnings.filterwarnings('ignore', module='sklearn')


def gwet_ac2(a, b, weights='quadratic', scale_min=None, scale_max=None):
    """Gwet's AC2 for two raters on ordinal data.

    Robust to the kappa paradox under marginal skew. Reference: Gwet
    (2014), Handbook of Inter-Rater Reliability, 4th ed., Chapter 3.
    Pass scale_min and scale_max explicitly when the observed data
    does not cover the full rating scale (e.g. all observations in {4,5}
    on a 1-5 Likert: pass scale_min=1, scale_max=5).
    """
    a = np.asarray(a, dtype=int)
    b = np.asarray(b, dtype=int)
    assert len(a) == len(b)
    n = len(a)
    if scale_min is None:
        scale_min = int(min(a.min(), b.min()))
    if scale_max is None:
        scale_max = int(max(a.max(), b.max()))
    cats = np.arange(scale_min, scale_max + 1)
    q = len(cats)
    if q < 2:
        return 1.0 if np.all(a == b) else 0.0
    cat_to_idx = {int(c): i for i, c in enumerate(cats)}

    if weights == 'quadratic':
        idx = np.arange(q)
        W = 1 - (np.subtract.outer(idx, idx) / (q - 1)) ** 2
    elif weights == 'linear':
        idx = np.arange(q)
        W = 1 - np.abs(np.subtract.outer(idx, idx)) / (q - 1)
    elif weights is None:
        W = np.eye(q)
    else:
        raise ValueError(f"Unknown weights: {weights}")

    p_o = np.mean([W[cat_to_idx[int(ai)], cat_to_idx[int(bi)]]
                   for ai, bi in zip(a, b)])

    pi = np.zeros(q)
    for i, c in enumerate(cats):
        pi[i] = (np.sum(a == c) + np.sum(b == c)) / (2 * n)

    # Gwet (2014) AC2 chance agreement: the TOTAL weight sum is factored
    # across the q(q-1) ordered category pairs. This reduces to AC1's
    # (1/(q-1)) * sum_k pi_k(1-pi_k) when weights are the identity.
    Tw = W.sum()
    p_e = Tw * np.sum(pi * (1 - pi)) / (q * (q - 1))

    if p_e >= 1.0:
        return 1.0
    return (p_o - p_e) / (1 - p_e)


def agreement_within(a, b, tolerance=1):
    """Fraction of pairs where |a - b| <= tolerance."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return np.mean(np.abs(a - b) <= tolerance)


def compute_ordinal_agreement(df, score_columns,
                              group_cols=('Sentenza', 'Questioni estratte'),
                              annotator_col='annotator',
                              relevance_col=COL_PRESENT,
                              scale_min=1, scale_max=5):
    """
    For each score column, compute agreement metrics between two annotators.
    Excludes issues flagged as hallucinated (relevance_col == False) by either
    annotator.
    """

    # --- Step 1: Filter out hallucinated issues ---
    def both_present(group):
        return group[relevance_col].all()

    df_valid = (
        df.groupby(list(group_cols))
        .filter(both_present)
    )

    # --- Step 2: Pivot to get one row per issue, one column per annotator ---
    results = []

    for col in score_columns:
        pivoted = (
            df_valid
            .dropna(subset=[col])
            .pivot_table(index=list(group_cols), columns=annotator_col,
                         values=col, aggfunc='first')
            .dropna()
        )

        if len(pivoted) < 2:
            results.append({'column': col, 'n': len(pivoted),
                            'note': 'too few pairs'})
            continue

        a = pivoted['A1'].values
        p = pivoted['A2'].values

        # Pick the (smin, smax) bounds gwet_ac2 will use. Three cases:
        #   1. Binary 0/1 column: bounds (0, 1).
        #   2. Count/sum column whose observed values fall outside the
        #      default Likert band: use the observed min/max so every value
        #      appears in cats.
        #   3. Likert column saturated inside (scale_min, scale_max): keep
        #      the full declared scale so chance-correction reflects the
        #      number of categories actually available to raters.
        combined = np.unique(np.concatenate([a, p])).astype(int)
        if set(combined).issubset({0, 1}):
            smin, smax = 0, 1
        elif combined.min() < scale_min or combined.max() > scale_max:
            smin, smax = int(combined.min()), int(combined.max())
        else:
            smin, smax = scale_min, scale_max

        results.append({
            'column': col,
            'n': len(pivoted),
            'mean_a1': np.mean(a),
            'mean_a2': np.mean(p),
            'exact_agreement': np.mean(a == p),
            'within_1_agreement': agreement_within(a, p, tolerance=1),
            'gwet_ac2_quad': gwet_ac2(a, p, weights='quadratic',
                                      scale_min=smin, scale_max=smax),
            'kappa_linear': cohen_kappa_score(p, a, weights='linear'),
            'kappa_quadratic': cohen_kappa_score(p, a, weights='quadratic'),
            'pearson_r': (np.corrcoef(a, p)[0, 1]
                          if np.std(a) > 0 and np.std(p) > 0 else np.nan),
        })

    return pd.DataFrame(results)

## Inter-annotator agreement and LLM averages — Table `tab:quality_scores`

IAA is computed on the shared subset (N=20 judgments, issues marked present by both
annotators). The LLM average uses the combined N=50 frame (shared issues contribute the
mean of the two annotators' scores). For outcome satisfaction both annotators assigned 5
to every issue, so variance-based statistics (κ, r) are undefined.

In [6]:
# Shared-subset frame with one row per (issue, annotator)
df_shared = pd.concat([df_a.assign(annotator='A1'), df_p.assign(annotator='A2')],
                      ignore_index=True)
df_shared = df_shared[df_shared['Sentenza'].isin(common_sentenze)]

results_df = compute_ordinal_agreement(df_shared, score_columns)

# LLM averages on the full corpus (joint-present issues of the combined frame)
pres_comb = df_combined[df_combined['is_present_joint'] == True]
llm_means = {col: pres_comb[col].mean() for col in score_columns}

table_rows = []
for disp, col in DIMENSIONS:
    r = results_df[results_df['column'] == col].iloc[0]
    table_rows.append({
        'Dimension': disp,
        'Mean A1': f"{r['mean_a1']:.2f}",
        'Mean A2': f"{r['mean_a2']:.2f}",
        'EA':  f"{r['exact_agreement']:.1%}",
        'W1':  f"{r['within_1_agreement']:.1%}",
        'AC2': f"{r['gwet_ac2_quad']:.2f}",
        'LLM avg (N=50)': f"{llm_means[col]:.2f}",
    })

print(f"Issues in the IAA comparison: {results_df['n'].iloc[0]}")
print(f"Issues in the LLM average:   {len(pres_comb)}")
print()
print('Table tab:quality_scores:')
print(pd.DataFrame(table_rows).set_index('Dimension').to_string())

Issues in the IAA comparison: 28
Issues in the LLM average:   74

Table tab:quality_scores:
                                Mean A1 Mean A2      EA      W1   AC2 LLM avg (N=50)
Dimension                                                                           
Issue <text> generality            4.75    4.32   64.3%   92.9%  0.93           4.61
Issue <text> legal language        4.89    4.79   89.3%  100.0%  0.99           4.84
<summary> correctness              4.79    4.86   85.7%   96.4%  0.97           4.74
<summary> form                     4.86    4.93   78.6%  100.0%  0.98           4.82
<summary> completeness             4.86    4.86   85.7%   92.9%  0.97           4.85
<summary> satisfaction             4.89    4.82   85.7%   92.9%  0.97           4.77
<factual_premises> satisfaction    4.79    4.54   82.1%   89.3%  0.94           4.68
<judge_reasoning> satisfaction     4.93    4.82   89.3%   92.9%  0.98           4.78
<issue_outcome> satisfaction       5.00    5.00  100.0%  1

## Cohen's κ and Pearson r — Appendix Table `tab:quality_scores_kappa`

Weighted Cohen's κ (linear and quadratic) and Pearson correlation for the same dimensions,
plus the citation-reason counts (per-issue number of faithful reasons, an integer count
that is well spread rather than saturated, so the kappa paradox does not arise there).

In [7]:
fmt_k = lambda v: '--' if pd.isna(v) else f'{v:.2f}'

kappa_rows = []
for disp, col in DIMENSIONS:
    r = results_df[results_df['column'] == col].iloc[0]
    kappa_rows.append({'Dimension': disp,
                       'kappa_lin': fmt_k(r['kappa_linear']),
                       'kappa_quad': fmt_k(r['kappa_quadratic']),
                       'pearson_r': fmt_k(r['pearson_r'])})

# Citation reasons: per-issue integer count of faithful reasons (0..N_i).
df_valid = (df_shared.groupby(['Sentenza', 'Questioni estratte'])
                     .filter(lambda g: g[COL_PRESENT].all()))
piv = (df_valid.dropna(subset=[COL_SCORE_MOTIVI])
              .pivot_table(index=['Sentenza', 'Questioni estratte'], columns='annotator',
                           values=COL_SCORE_MOTIVI, aggfunc='first')
              .dropna())
s_a, s_p = piv['A1'].astype(int).values, piv['A2'].astype(int).values
kappa_rows.append({'Dimension': 'Citation reasons (faithful count)',
                   'kappa_lin':  fmt_k(cohen_kappa_score(s_p, s_a, weights='linear')),
                   'kappa_quad': fmt_k(cohen_kappa_score(s_p, s_a, weights='quadratic')),
                   'pearson_r':  fmt_k(np.corrcoef(s_a, s_p)[0, 1])})

print('Table tab:quality_scores_kappa:')
print(pd.DataFrame(kappa_rows).set_index('Dimension').to_string())

Table tab:quality_scores_kappa:
                                  kappa_lin kappa_quad pearson_r
Dimension                                                       
Issue <text> generality                0.40       0.48      0.63
Issue <text> legal language            0.63       0.74      0.78
<summary> correctness                  0.35       0.34      0.38
<summary> form                        -0.11      -0.11     -0.11
<summary> completeness                 0.18       0.08      0.08
<summary> satisfaction                 0.18       0.08      0.09
<factual_premises> satisfaction        0.45       0.40      0.46
<judge_reasoning> satisfaction         0.24       0.13      0.17
<issue_outcome> satisfaction             --         --        --
Citation reasons (faithful count)      0.91       0.85      0.86
